<a href="https://colab.research.google.com/github/liguangxun0106/Mujoco_tutorial/blob/main/tutorial/02_rendering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/02_rendering.ipynb)

# Tutorial 2: Rendering

Learn to render scenes, configure cameras, and create videos.

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media

## Basic Rendering with `mujoco.Renderer`

The `Renderer` class handles offscreen rendering — perfect for notebooks and headless servers.

In [ ]:
xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1.5" dir="0 0 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    <geom name="red_box" type="box" size=".2 .2 .2" pos="0 0 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" type="sphere" pos=".4 .3 .1" size=".1" rgba="0 1 0 1"/>
    <geom name="blue_cylinder" type="cylinder" pos="-.3 .2 .15" size=".08 .15" rgba="0 0 1 1"/>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

# Render
renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data)
media.show_image(renderer.render())

## Camera Configuration

You can control the viewpoint by configuring the camera.

In [ ]:
# Create a camera and set its parameters
camera = mujoco.MjvCamera()
camera.azimuth = 135
camera.elevation = -20
camera.distance = 2.0
camera.lookat[:] = [0, 0, 0.2]

renderer.update_scene(data, camera=camera)
media.show_image(renderer.render())

In [ ]:
# Render from multiple viewpoints
images = []
for azimuth in [0, 90, 180, 270]:
    camera.azimuth = azimuth
    renderer.update_scene(data, camera=camera)
    images.append(renderer.render().copy())

media.show_images(images, titles=['Front', 'Right', 'Back', 'Left'])

## Fixed Cameras in XML

You can also define cameras directly in the model:

In [ ]:
xml_with_cameras = """
<mujoco>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    <geom type="box" size=".2 .2 .2" pos="0 0 .2" rgba="1 0 0 1"/>
    <camera name="top_down" pos="0 0 2" xyaxes="1 0 0 0 1 0"/>
    <camera name="side" pos="1.5 0 0.5" xyaxes="0 1 0 0 0 1"/>
  </worldbody>
</mujoco>
"""

model2 = mujoco.MjModel.from_xml_string(xml_with_cameras)
data2 = mujoco.MjData(model2)
mujoco.mj_forward(model2, data2)

renderer2 = mujoco.Renderer(model2, height=300, width=400)

images = []
for cam_name in ['top_down', 'side']:
    cam_id = mujoco.mj_name2id(model2, mujoco.mjtObj.mjOBJ_CAMERA, cam_name)
    renderer2.update_scene(data2, camera=cam_id)
    images.append(renderer2.render().copy())

media.show_images(images, titles=['Top-down', 'Side'])
renderer2.close()

## Creating Videos

Render frames during simulation and combine into a video:

In [ ]:
xml_ball = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom type="plane" size="3 3 0.1" rgba=".9 .9 .9 1"/>
    <body pos="0 0 2">
      <joint type="free"/>
      <geom type="sphere" size="0.15" rgba="0.2 0.5 1 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model3 = mujoco.MjModel.from_xml_string(xml_ball)
data3 = mujoco.MjData(model3)
renderer3 = mujoco.Renderer(model3, height=480, width=640)

# Give initial velocity
data3.qvel[0] = 3.0  # horizontal

frames = []
fps = 60
duration = 2.0

while data3.time < duration:
    mujoco.mj_step(model3, data3)
    if len(frames) < data3.time * fps:
        renderer3.update_scene(data3)
        frames.append(renderer3.render().copy())

renderer3.close()
media.show_video(frames, fps=fps)

## Depth and Segmentation Rendering

In [ ]:
model4 = mujoco.MjModel.from_xml_string(xml)
data4 = mujoco.MjData(model4)
mujoco.mj_forward(model4, data4)

renderer4 = mujoco.Renderer(model4, height=480, width=640)

# RGB
renderer4.update_scene(data4)
rgb = renderer4.render().copy()

# Depth
renderer4.enable_depth_rendering()
renderer4.update_scene(data4)
depth = renderer4.render().copy()
renderer4.disable_depth_rendering()

# Segmentation
renderer4.enable_segmentation_rendering()
renderer4.update_scene(data4)
seg = renderer4.render()[:, :, 0].copy()  # geom IDs in first channel
renderer4.disable_segmentation_rendering()

renderer4.close()

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(rgb); axes[0].set_title('RGB')
axes[1].imshow(depth, cmap='viridis'); axes[1].set_title('Depth')
axes[2].imshow(seg, cmap='tab10'); axes[2].set_title('Segmentation')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()